# 01 · Obtención de datos
**Taller de Ciencia de Datos (EIN146A25)** · Riesgo de reprobación y deserción escolar

Este cuaderno (a) verifica qué archivos hay, (b) los descarga por código si se entregan enlaces,
(c) audita columnas y volumen de cada archivo completo y (d) genera las **muestras versionadas**
(`data/sample/`, 5 % de los estudiantes) con las que cualquiera puede reproducir todo el proyecto
desde un clon vacío del repositorio.

**Salidas:** `results/volumen_archivos.csv`, `results/columnas.json`, `data/sample/muestra_*.csv.gz`.

In [ ]:
import sys, warnings
from pathlib import Path
RAIZ = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(RAIZ / 'src'))
import numpy as np, pandas as pd
import proyecto as P
warnings.filterwarnings('ignore')
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)
print('Raíz del proyecto:', P.RAIZ)

## 1. Fuentes y licencia
Datos abiertos del Centro de Estudios del Ministerio de Educación (MINEDUC), <https://datosabiertos.mineduc.cl/>,
sección *Estudiantes → Educación básica y media*: **Matrícula por estudiante** y **Rendimiento por estudiante**
(años 2023, 2024 y 2025). Licencia: datos abiertos del Estado de Chile, uso con citación de la fuente.
El identificador de estudiante es `MRUN` (anonimizado).

> Se evaluó la Encuesta ENDDEIE (Desarrollo Digital e Innovación Educativa) y se **descartó**: es una encuesta
> muestral sobre tecnología en los establecimientos, sin identificador de estudiante, por lo que no puede unirse
> a nivel de estudiante.

## 2. Inventario de archivos

In [ ]:
for tipo in ('rend', 'mat'):
    for a in P.ANIOS:
        r = P.ubicar_archivo(tipo, a)
        print(f"{'Rendimiento' if tipo == 'rend' else 'Matrícula':12s} {a}:", r if r else 'NO ENCONTRADO')

## 3. Descarga por código (opcional)
Pegue en `URLS` los **enlaces directos** de descarga que entrega el portal (no se incluyen aquí porque cambian
con cada publicación). Admite `.zip` y `.rar` (para `.rar` hay que tener `unrar` o `7z` instalado;
en Colab: `!apt-get -qq install unrar p7zip-full`). Si `URLS` está vacío, esta celda no hace nada y se usan
los archivos que ya estén en `data/raw/` o, en su defecto, las muestras de `data/sample/`.

In [ ]:
import urllib.request, zipfile, subprocess, shutil

URLS = {
    # 'rend_2023': 'https://.../Rendimiento-2023.zip',
    # 'mat_2023':  'https://.../Matricula-por-estudiante-2023.zip',
}

def descargar_y_extraer(url, destino=P.DIR_RAW):
    archivo = destino / url.split('/')[-1].split('?')[0]
    print('Descargando', url)
    urllib.request.urlretrieve(url, archivo)
    if archivo.suffix.lower() == '.zip':
        zipfile.ZipFile(archivo).extractall(destino)
    elif archivo.suffix.lower() == '.rar':
        herramienta = shutil.which('unrar') or shutil.which('7z')
        if herramienta is None:
            raise RuntimeError('Instale unrar o 7z para abrir .rar')
        cmd = [herramienta, 'x', '-o+', str(archivo), str(destino) + '/'] if 'unrar' in herramienta else [herramienta, 'x', f'-o{destino}', '-y', str(archivo)]
        subprocess.run(cmd, check=True)
    print('Listo:', archivo.name)

for clave, url in URLS.items():
    descargar_y_extraer(url)
if not URLS:
    print('URLS vacío: se usan los archivos ya presentes.')

## 4. Auditoría de columnas y volumen (archivos completos) y creación de muestras
Para cada archivo completo en `data/raw/` se registran sus columnas y su número de filas, y se guarda una
**muestra del 5 %** (estudiantes con `MRUN` múltiplo de 20, de modo que el mismo estudiante queda en todos los años).
Si no hay archivos completos, se informa y se usan las muestras existentes.

In [ ]:
import json
volumen, columnas = [], {}
for tipo in ('rend', 'mat'):
    for a in P.ANIOS:
        ruta = P.ubicar_archivo(tipo, a)
        if ruta is None or ruta.parent != P.DIR_RAW:
            print(f'{tipo} {a}: sin archivo completo en data/raw (se omite)')
            continue
        sep = P._sep(ruta)
        columnas[ruta.name] = [P._norm(c) for c in pd.read_csv(ruta, sep=sep, encoding='latin1', nrows=0).columns]
        est = {}
        cols = ['MRUN', 'AGNO', 'GEN_ALU', 'COD_ENSE', 'COD_ENSE2', 'PROM_GRAL', 'SIT_FIN', 'SIT_FIN_R', 'ASISTENCIA']
        muestra = P.leer_csv(ruta, cols, mrun_modulo=20, estadisticas=est)
        muestra.to_csv(P.DIR_MUESTRA / f'muestra_{tipo}_{a}.csv.gz', index=False, compression='gzip')
        volumen.append({'archivo': ruta.name, 'tipo': tipo, 'anio': a, 'filas_totales': est['filas'],
                        'filas_muestra': len(muestra), 'sit_fin': est.get('sit_fin')})
        print(f"{ruta.name}: {est['filas']:,} filas -> muestra {len(muestra):,}")
if volumen:
    pd.DataFrame(volumen).to_csv(P.DIR_RES / 'volumen_archivos.csv', index=False)
    P.guardar_json(columnas, 'columnas.json')
    display(pd.DataFrame(volumen))

## 5. Qué sigue
`02-preprocesamiento.ipynb` lee `data/raw/` (o `data/sample/`), limpia, integra Matrícula y Rendimiento y exporta
`data/processed/dataset_final`.